# Defining Cofactors

You can define cofactors, any small molecules that are not the bound ligand, using the openfe CLI during the planning phase or directly using the Python API.

Similar to `ligand` definition, `cofactors` can be loaded from an SDF file format.

In [2]:
%matplotlib inline
import openfe

## CLI: define cofactors during ``openfe plan-*``

In [13]:
! openfe plan-rbfe-network -M ligands.sdf -p protein.pdb --cofactors cofactors.sdf

/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=60788) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Usage: openfe plan-rbfe-network [OPTIONS]
Try 'openfe plan-rbfe-network -h' for help.

Error: Invalid value for '-M' / '--molecules': Path 'ligands.sdf' does not exist.


## Python API: define cofactors in the ``ChemicalSystem``s 

### Load the ligands and cofactors

In [8]:
from rdkit import Chem
from openff.toolkit import Molecule

# Create a list of OpenFF molecules
offmols = [
    Molecule.from_rdkit(m)
    for m in Chem.SDMolSupplier("eg5_ligands_positive.sdf", removeHs=False)
]

# Assign partial charges via AM1BCC
for mol in offmols:
    mol.assign_partial_charges('am1bcc', use_conformers=mol.conformers)

# Now we convert them to SmallMoleculeComponent
ligands = [openfe.SmallMoleculeComponent.from_openff(mol) for mol in offmols]

/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/openff/toolkit/utils/toolkit_registry.py:262: UserWarning: `NAGLToolkitWrapper.assign_partial_charges` was passed optional argument `use_conformers` which will not be used. OpenFF NAGL does not generate conformers as part of assigning partial charges.
  return method(*args, **kwargs)
/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/openff/toolkit/utils/toolkit_registry.py:262: UserWarning: `NAGLToolkitWrapper.assign_partial_charges` was passed optional argument `use_conformers` which will not be used. OpenFF NAGL does not generate conformers as part of assigning partial charges.
  return method(*args, **kwargs)
/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/openff/toolkit/utils/toolkit_registry.py:262: UserWarning: `NAGLToolkitWrapper.assign_partial_charges` was passed optional argument `use_conformers` which will not be used. OpenFF NAGL does not ge

In [7]:
# load in the cofactor from eg5_cofactors.sdf, here we know that there is only one cofactor in the file
# We also go through OpenFF to assign partial charges
offmol = Molecule.from_rdkit(Chem.MolFromMolFile("eg5_cofactor.sdf", removeHs=False))
offmol.assign_partial_charges('am1bcc', use_conformers=offmol.conformers)
cofactor = openfe.SmallMoleculeComponent.from_openff(offmol)

/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/openff/toolkit/utils/toolkit_registry.py:262: UserWarning: `NAGLToolkitWrapper.assign_partial_charges` was passed optional argument `use_conformers` which will not be used. OpenFF NAGL does not generate conformers as part of assigning partial charges.
  return method(*args, **kwargs)


### Create the Network

In [ ]:
mapper = openfe.LomapAtomMapper(max3d=1.0, element_change=False, threed=True)
scorer = openfe.lomap_scorers.default_lomap_score
network_planner = openfe.ligand_network_planning.generate_minimal_spanning_network

### Create the Chemical Systems

In [9]:
# defaults are water with NaCl at 0.15 M
solvent = openfe.SolventComponent()

# load in the EG5 protein
protein = openfe.ProteinComponent.from_pdb_file("./eg5_protein.pdb")

In [10]:
# load in the cofactor from eg5_cofactors.sdf, here we know that there is only one cofactor in the file
# We also go through OpenFF to assign partial charges
offmol = Molecule.from_rdkit(Chem.MolFromMolFile("eg5_cofactor.sdf", removeHs=False))
offmol.assign_partial_charges('am1bcc', use_conformers=offmol.conformers)
cofactor = openfe.SmallMoleculeComponent.from_openff(offmol)

/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/openff/toolkit/utils/toolkit_registry.py:262: UserWarning: `NAGLToolkitWrapper.assign_partial_charges` was passed optional argument `use_conformers` which will not be used. OpenFF NAGL does not generate conformers as part of assigning partial charges.
  return method(*args, **kwargs)


In [11]:
# Note: if you had multiple cofactors you could add any additional cofactor
# as additional dictionary entries in both of the ChemicalSystem definitions
systemA = openfe.ChemicalSystem({
    'ligand': mapping.componentA,
    'solvent': solvent,
    'protein': protein,
    'cofactor': cofactor,
})
systemB = openfe.ChemicalSystem({
    'ligand': mapping.componentB,
    'solvent': solvent,
    'protein': protein,
    'cofactor': cofactor,
})

NameError: name 'mapping' is not defined

In [ ]:
transformations = []
for mapping in ligand_network.edges:
    for leg in ['solvent', 'complex']:
        # use the solvent and protein created above
        sysA_dict = {'ligand': mapping.componentA,
                     'solvent': solvent}
        sysB_dict = {'ligand': mapping.componentB,
                     'solvent': solvent}
        
        if leg == 'complex':
            sysA_dict['protein'] = protein
            sysA_dict['cofactor'] = cofactor
            sysB_dict['protein'] = protein
            sysB_dict['cofactor'] = cofactor
        
        # we don't have to name objects, but it can make things (like filenames) more convenient
        sysA = openfe.ChemicalSystem(sysA_dict, name=f"{mapping.componentA.name}_{leg}")
        sysB = openfe.ChemicalSystem(sysB_dict, name=f"{mapping.componentB.name}_{leg}")
        
        prefix = "easy_rbfe_"  # prefix is only to exactly reproduce CLI
        
        transformation = openfe.Transformation(
            stateA=sysA,
            stateB=sysB,
            mapping={'ligand': mapping},
            protocol=protocol,  # use protocol created above
            name=f"{prefix}{sysA.name}_{sysB.name}"
        )
        transformations.append(transformation)

network = openfe.AlchemicalNetwork(transformations)